In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Load saved dataset
df = pd.read_csv('../data/raw/german_credit.csv')

# Separate features and target
X = df.drop('credit_risk', axis=1)
y = df['credit_risk']

# Identify feature types
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical: {len(cat_cols)} features")
print(f"Numerical:   {len(num_cols)} features")
print(f"Target:      {y.value_counts().to_dict()}")

In [ ]:
# Label encode all categorical features
# (LabelEncoder for tree-based models — they handle ordinal encoding well)
le_dict = {}
X_encoded = X.copy()

for col in cat_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col])
    le_dict[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print(f"\nEncoded shape: {X_encoded.shape}")
print(f"All dtypes now numeric: {X_encoded.dtypes.value_counts().to_dict()}")
print(f"\nFirst 3 rows after encoding:")
X_encoded.head(3)

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Save encoded features + target
df_encoded = X_encoded.copy()
df_encoded['credit_risk'] = y.values
df_encoded.to_csv('../data/processed/german_credit_encoded.csv', index=False)

print(f"Saved encoded dataset: {df_encoded.shape}")
print(f"Location: data/processed/german_credit_encoded.csv")

In [ ]:
# Train/test split — 80/20 stratified by target
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print("BEFORE SMOTE:")
print(f"  Training set: {X_train.shape[0]} samples")
print(f"    Good (0): {(y_train==0).sum()} ({(y_train==0).mean():.1%})")
print(f"    Bad  (1): {(y_train==1).sum()} ({(y_train==1).mean():.1%})")
print(f"  Test set:     {X_test.shape[0]} samples")
print(f"    Good (0): {(y_test==0).sum()} ({(y_test==0).mean():.1%})")
print(f"    Bad  (1): {(y_test==1).sum()} ({(y_test==1).mean():.1%})")

In [ ]:
# Apply SMOTE to training data ONLY
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("AFTER SMOTE (training set only):")
print(f"  Training set: {X_train_smote.shape[0]} samples")
print(f"    Good (0): {(y_train_smote==0).sum()} ({(y_train_smote==0).mean():.1%})")
print(f"    Bad  (1): {(y_train_smote==1).sum()} ({(y_train_smote==1).mean():.1%})")
print(f"  Test set:     {X_test.shape[0]} samples (UNCHANGED)")
print(f"    Good (0): {(y_test==0).sum()}")
print(f"    Bad  (1): {(y_test==1).sum()}")
print(f"\n✓ SMOTE applied to training set only — no data leakage")

In [ ]:
# Before/after SMOTE comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Before
counts_before = y_train.value_counts()
ax1.bar(['Good (0)', 'Bad (1)'], counts_before.values, color=['#2E8BC0', '#C0392B'])
for i, v in enumerate(counts_before.values):
    ax1.text(i, v+5, str(v), ha='center', fontweight='bold')
ax1.set_title('Before SMOTE')
ax1.set_ylabel('Count')
ax1.set_ylim(0, max(counts_before.values)*1.15)

# After
counts_after = pd.Series(y_train_smote).value_counts()
ax2.bar(['Good (0)', 'Bad (1)'], counts_after.values, color=['#2E8BC0', '#C0392B'])
for i, v in enumerate(counts_after.values):
    ax2.text(i, v+5, str(v), ha='center', fontweight='bold')
ax2.set_title('After SMOTE')
ax2.set_ylabel('Count')
ax2.set_ylim(0, max(counts_after.values)*1.15)

plt.suptitle('Figure 5: Class Distribution Before and After SMOTE', fontsize=14)
plt.tight_layout()
plt.savefig('../shap_plots/smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()